## Dividend Based Investing


In [68]:
# importing necessary libraries
import pandas as pd
import numpy as np
import math
import yfinance as yf
from scipy import stats

In [69]:
#loading nifty data set
tickers=pd.read_csv(r"dataset/ind_nifty50list.csv")
tickers.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,Adani Enterprises Ltd.,Metals & Mining,ADANIENT,EQ,INE423A01024
1,Adani Ports and Special Economic Zone Ltd.,Services,ADANIPORTS,EQ,INE742F01042
2,Apollo Hospitals Enterprise Ltd.,Healthcare,APOLLOHOSP,EQ,INE437A01024
3,Asian Paints Ltd.,Consumer Durables,ASIANPAINT,EQ,INE021A01026
4,Axis Bank Ltd.,Financial Services,AXISBANK,EQ,INE238A01034


In [70]:
#removing un required columns and adding a ticker column for yfinance
tickers['Symbol']=tickers['Symbol']+'.NS'
tickers=tickers.drop(['Industry','Series','ISIN Code'],axis=1)

Weighted scoring model for finding the final score


In [71]:
def dividend_df(tickers):
    col=[
        "Ticker","Dividend yield(%)","Dividend Rate","Payout Ratio(%)","5 year avg div yield(%)"
    ]
    dividend_df=pd.DataFrame(columns=col)
    for stock in tickers:
        ticker=yf.Ticker(stock)
        info=ticker.info

        div_yield=info.get("dividendYield",np.nan) * 100
        div_rate=info.get("dividendRate",np.nan)
        payout_ratio=info.get("payoutRatio",np.nan) *100
        five_year_avg=info.get("fiveYearAvgDividendYield",np.nan) *100
       # earning_growth=info.get("earningGrowth",np.nan) *100

        dividend_df.loc[len(dividend_df)]=[stock,div_yield,div_rate,payout_ratio,five_year_avg]
    # we created our data frame with divident data

    numeric_col=[
        "Dividend yield(%)","Dividend Rate","Payout Ratio(%)","5 year avg div yield(%)"
    ]

    for col in numeric_col:
        if col=="Payout Ratio(%)":
            dividend_df[col + "Normalized"]=1- ((dividend_df[col] - dividend_df[col].min())/(dividend_df[col].max() - dividend_df[col].min()))
        else:
            dividend_df[col + "Normalized"]=(dividend_df[col] - dividend_df[col].min())/(dividend_df[col].max() - dividend_df[col].min())
    return dividend_df


In [72]:
tickers_list=tickers['Symbol'].values.tolist()
div_df=dividend_df(tickers_list)
div_df.head()

,Ticker,Dividend yield(%),Dividend Rate,Payout Ratio(%),5 year avg div yield(%),Dividend yield(%)Normalized,Dividend RateNormalized,Payout Ratio(%)Normalized,5 year avg div yield(%)Normalized
0,ADANIENT.NS,5.0,1.30,2.77,5.0,0.004897,0.002924,0.971212,0.001351
1,ADANIPORTS.NS,39.0,7.50,12.52,51.0,0.038198,0.028822,0.869882,0.063514
2,APOLLOHOSP.NS,23.0,20.00,12.79,18.0,0.022527,0.081036,0.867075,0.018919
3,ASIANPAINT.NS,115.0,28.75,54.50,84.0,0.112635,0.117586,0.433590,0.108108
4,AXISBANK.NS,8.0,1.00,1.04,9.0,0.007835,0.001671,0.989191,0.006757


In [73]:
#checking and handling null values
div_df.isna().sum()


Ticker                               0
Dividend yield(%)                    0
Dividend Rate                        1
Payout Ratio(%)                      0
5 year avg div yield(%)              3
Dividend yield(%)Normalized          0
Dividend RateNormalized              1
Payout Ratio(%)Normalized            0
5 year avg div yield(%)Normalized    3
dtype: int64

In [74]:
# assigning weights
weights={
        "Dividend yield(%)Normalized": 0.3,
        "Dividend RateNormalized": 0.25,
        "Payout Ratio(%)Normalized":0.25 ,
        "5 year avg div yield(%)Normalized": 0.2
    }
div_df["Div_Score"]=div_df[[col for col in weights.keys()]].mul(list(weights.values())).sum(axis=1)
div_df.head()

,Ticker,Dividend yield(%),Dividend Rate,Payout Ratio(%),5 year avg div yield(%),Dividend yield(%)Normalized,Dividend RateNormalized,Payout Ratio(%)Normalized,5 year avg div yield(%)Normalized,Div_Score
0,ADANIENT.NS,5.0,1.30,2.77,5.0,0.004897,0.002924,0.971212,0.001351,0.245273
1,ADANIPORTS.NS,39.0,7.50,12.52,51.0,0.038198,0.028822,0.869882,0.063514,0.248838
2,APOLLOHOSP.NS,23.0,20.00,12.79,18.0,0.022527,0.081036,0.867075,0.018919,0.247570
3,ASIANPAINT.NS,115.0,28.75,54.50,84.0,0.112635,0.117586,0.433590,0.108108,0.193206
4,AXISBANK.NS,8.0,1.00,1.04,9.0,0.007835,0.001671,0.989191,0.006757,0.251418


In [75]:
def get_price(ticker_list):
    data=yf.download(tickers_list,period='1d',group_by='ticker',auto_adjust=False)
    stocks_data=[]
    for ticker in tickers_list:
        latest_price=data[ticker]['Close'].iloc[-1]
        stocks_data.append(
            {
                 "Latest Price": latest_price
            }
        )
    stock_df=pd.DataFrame(stocks_data)
    return stock_df

In [76]:
#adding price column and checking how many stocks can be bought
div_df['Price']=get_price(tickers_list)
div_df.head()

[*********************100%***********************]  50 of 50 completed


,Ticker,Dividend yield(%),Dividend Rate,Payout Ratio(%),5 year avg div yield(%),Dividend yield(%)Normalized,Dividend RateNormalized,Payout Ratio(%)Normalized,5 year avg div yield(%)Normalized,Div_Score,Price
0,ADANIENT.NS,5.0,1.30,2.77,5.0,0.004897,0.002924,0.971212,0.001351,0.245273,2706.300049
1,ADANIPORTS.NS,39.0,7.50,12.52,51.0,0.038198,0.028822,0.869882,0.063514,0.248838,1942.699951
2,APOLLOHOSP.NS,23.0,20.00,12.79,18.0,0.022527,0.081036,0.867075,0.018919,0.247570,8549.500000
3,ASIANPAINT.NS,115.0,28.75,54.50,84.0,0.112635,0.117586,0.433590,0.108108,0.193206,2509.300049
4,AXISBANK.NS,8.0,1.00,1.04,9.0,0.007835,0.001671,0.989191,0.006757,0.251418,1310.699951


In [77]:
div_df=div_df.sort_values(by=["Div_Score"],ascending=False)
div_df=div_df.head(10)
portfolio_size=int(input("Enter the amount to invest : "))
position_size=portfolio_size/len(div_df.index)
div_df['Number of shares to buy']=div_df['Price'].apply(lambda price: math.floor(
    position_size/price
))

# reseting indexes

div_df.reset_index(drop=True)
div_df

,Ticker,Dividend yield(%),Dividend Rate,Payout Ratio(%),5 year avg div yield(%),Dividend yield(%)Normalized,Dividend RateNormalized,Payout Ratio(%)Normalized,5 year avg div yield(%)Normalized,Div_Score,Price,Number of shares to buy
12,COALINDIA.NS,1021.0,41.00,39.250000,744.0,1.000000,0.168755,0.592081,1.000000,0.690209,401.500000,24
35,ONGC.NS,567.0,13.25,33.690000,510.0,0.555338,0.052840,0.649865,0.683784,0.479034,233.500000,42
6,BAJAJ-AUTO.NS,137.0,150.00,32.000000,178.0,0.134182,0.624060,0.667429,0.235135,0.410154,10979.000000,0
36,POWERGRID.NS,465.0,12.00,51.660000,434.0,0.455436,0.047619,0.463105,0.581081,0.380528,257.950012,38
33,NTPC.NS,365.0,12.00,29.350000,296.0,0.357493,0.047619,0.694970,0.394595,0.371814,328.750000,30
49,ULTRACEMCO.NS,196.0,240.00,80.250000,56.0,0.191969,1.000000,0.165974,0.070270,0.363138,12275.000000,0
31,MARUTI.NS,108.0,140.00,32.720003,79.0,0.105779,0.582289,0.659946,0.101351,0.362563,12928.000000,0
24,INFY.NS,451.0,52.00,67.780000,239.0,0.441724,0.214703,0.295573,0.317568,0.323600,1154.099976,8
40,SBIN.NS,167.0,17.35,17.709999,145.0,0.163565,0.069967,0.815943,0.190541,0.308655,1042.000000,9
14,EICHERMOT.NS,107.0,82.00,33.310002,82.0,0.104799,0.340017,0.653814,0.105405,0.300979,7629.000000,1
